# AI Morph Ads — Colab / Kaggle runner

One product image → four ad creatives (Luxury / Minimal / Urban / Nostalgic).

**Runtime:** GPU (T4 is enough — 15 GB VRAM).

This notebook:
1. Clones ComfyUI + required custom nodes
2. Clones *this* project and installs Python deps
3. Downloads all model weights (base models + 4 style LoRAs) into `ComfyUI/models/...`
4. Starts ComfyUI as a background server
5. Runs `scripts/run_pipeline.py` for a product image you upload

> **LoRAs** are now auto-downloaded from HuggingFace by `scripts/download_models.py`. If you want to override them with your own `.safetensors` from CivitAI, either run the download step with `--skip-loras` and drop your files manually into `ComfyUI/models/loras/`, or edit `configs/variants.yaml` to point at different filenames.

## 1. Verify GPU

In [1]:
!nvidia-smi

Thu Apr 23 16:15:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             30W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone ComfyUI + custom nodes + this project

In [2]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path('/content')
os.chdir(WORK)
print('Working dir:', WORK)

if not (WORK / 'ComfyUI').exists():
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git

NODES = WORK / 'ComfyUI' / 'custom_nodes'
NODES.mkdir(parents=True, exist_ok=True)

for name, repo in [
    ('ComfyUI-Manager',           'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI_IPAdapter_plus',    'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('comfyui_controlnet_aux',    'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
]:
    target = NODES / name
    if not target.exists():
        print(f'git clone {name}')
        subprocess.run(['git', 'clone', '--depth', '1', repo, str(target)], check=True)

print('\nInstalling ComfyUI requirements...')
!pip install -q -r ComfyUI/requirements.txt

print('Installing custom-node requirements...')
for name in ['ComfyUI-Manager', 'ComfyUI_IPAdapter_plus', 'comfyui_controlnet_aux']:
    req = NODES / name / 'requirements.txt'
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)

Working dir: /content

Installing ComfyUI requirements...
Installing custom-node requirements...


In [23]:
# !rm -rf /content/AI_Morph_Ads_with_ComfyUI
!git clone https://github.com/seemyoon/AI_Morph_Ads_with_ComfyUI.git

Cloning into 'AI_Morph_Ads_with_ComfyUI'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 50 (delta 15), reused 46 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 1.63 MiB | 5.35 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [3]:
PROJECT = WORK / 'AI_Morph_Ads_with_ComfyUI'
assert PROJECT.exists(), f'Upload / clone the project to {PROJECT}'

!pip install -q -r "$PROJECT/requirements.txt"

In [4]:
print('WORK: ', WORK)
print("PROJECT: ",PROJECT)

WORK:  /content
PROJECT:  /content/AI_Morph_Ads_with_ComfyUI


## 3. Download model weights

~10 GB total with SDXL. First run takes 5–10 min on a Colab T4; subsequent sessions skip anything already on disk.

Optional but recommended: set `HF_TOKEN` in your Colab environment before this step to avoid HuggingFace throttling and increase download rate limits.

In [5]:
!python "$PROJECT/scripts/download_models.py" --comfy-root "$WORK/ComfyUI"

Target: /content/ComfyUI/models

[auth] No HF token found. Downloads will run unauthenticated (lower rate limits).
--> checkpoints/v1-5-pruned-emaonly.safetensors
  [skip] v1-5-pruned-emaonly.safetensors already present
--> vae/vae-ft-mse-840000-ema-pruned.safetensors
  [skip] vae-ft-mse-840000-ema-pruned.safetensors already present
--> controlnet/control_v11p_sd15_canny.pth
  [skip] control_v11p_sd15_canny.pth already present
--> controlnet/control_v11f1p_sd15_depth.pth
  [skip] control_v11f1p_sd15_depth.pth already present
--> ipadapter/ip-adapter-plus_sd15.safetensors
  [skip] ip-adapter-plus_sd15.safetensors already present
--> clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors
  [skip] CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors already present
--> checkpoints/sd_xl_base_1.0.safetensors
  [skip] sd_xl_base_1.0.safetensors already present
--> loras/epiCRealism_Luxury.safetensors
  [skip] epiCRealism_Luxury.safetensors already present
--> loras/neon_street_night.safetensors
  [

In [5]:
from pathlib import Path

files = list(Path('/content/ComfyUI/models').iterdir())
print([p.name for p in files])

['style_models', 'clip_vision', 'vae', 'checkpoints', 'latent_upscale_models', 'controlnet', 'clip', 'loras', 'diffusers', 'hypernetworks', 'embeddings', 'diffusion_models', 'ipadapter', 'unet', 'audio_encoders', 'configs', 'frame_interpolation', 'gligen', 'model_patches', 'upscale_models', 'text_encoders', 'photomaker', 'vae_approx']


In [6]:
from pathlib import Path

loras = list(Path('/content/ComfyUI/models/loras').iterdir())
print([p.name for p in loras])

['neon_street_night.safetensors', '.cache', 'film_kodak_portra_v3.safetensors', 'put_loras_here', 'epiCRealism_Luxury.safetensors', 'minimalist_studio_v2.safetensors']


## 4. Verify auto-downloaded LoRAs + upload product image

By default, `scripts/download_models.py` now auto-downloads the 4 style LoRAs from HuggingFace into `ComfyUI/models/loras/` and creates the expected alias for the Minimal variant.

Use the Colab file browser (left sidebar) to upload only:
- 1 product image (PNG/JPG) into `AI Morph Ads/inputs/`

Optional:
- If you want custom LoRAs, rerun the download step with `--skip-loras` and place your own `.safetensors` under `ComfyUI/models/loras/` (filenames must match `configs/variants.yaml`).
- Set `HF_TOKEN` in Colab environment variables to avoid HuggingFace throttling.

In [6]:
expected_loras = {
    'epiCRealism_Luxury.safetensors',
    'minimalist_studio_v2.safetensors',
    'neon_street_night.safetensors',
    'film_kodak_portra_v3.safetensors',
}

loras = sorted((WORK / 'ComfyUI' / 'models' / 'loras').glob('*.safetensors'))
found_loras = {p.name for p in loras}

print('LoRAs found:')
for p in loras:
    print(' -', p.name)

missing = sorted(expected_loras - found_loras)
if missing:
    print('\n[WARN] Missing expected LoRA files:')
    for name in missing:
        print(' -', name)
    print('If HF download was rate-limited, set HF_TOKEN and rerun the download cell.')
else:
    print('\n[OK] All expected LoRA files are present.')

inputs = sorted((PROJECT / 'inputs').glob('*'))
print('\nProduct images found:')
for p in inputs:
    if p.is_file():
        print(' -', p.name)

LoRAs found:
 - epiCRealism_Luxury.safetensors
 - film_kodak_portra_v3.safetensors
 - minimalist_studio_v2.safetensors
 - neon_street_night.safetensors

[OK] All expected LoRA files are present.

Product images found:
 - .gitkeep


## 5. Start ComfyUI as a background server

We launch ComfyUI on port 8188 and auto-detect whether `--medvram` is supported by your ComfyUI version. On newer builds where `--medvram` is removed, it starts without that flag.

This cell prints the Colab proxy URL immediately and streams the latest startup log lines while waiting, so you can see progress in real time (instead of wondering whether startup is stuck).

In [20]:
import subprocess, time, re, os, urllib.request

subprocess.run('pkill -f "python.*main.py" ; pkill -f cloudflared ; fuser -k 8188/tcp', shell=True, capture_output=True)
time.sleep(3)

subprocess.run('wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i ~/cloudflared-linux-amd64.deb', shell=True, capture_output=True)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd=str(WORK / 'ComfyUI'),
    stdout=open(WORK / 'comfyui.log', 'w'),
    stderr=subprocess.STDOUT,
    env=env,
)
print(f'[1/3] ComfyUI started (PID {comfy.pid}), waiting for port 8188...')

import socket
for i in range(180):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    if sock.connect_ex(('127.0.0.1', 8188)) == 0:
        sock.close()
        print('[2/3] ComfyUI is ready on port 8188!')
        break
    sock.close()
    time.sleep(1)
else:
    raise RuntimeError('ComfyUI did not start. Run: !cat /content/comfyui.log')

tunnel_log = WORK / 'cloudflared.log'
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=open(tunnel_log, 'w'),
    stderr=subprocess.STDOUT,
)
print('[3/3] Cloudflared started, waiting for tunnel URL...')

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    try:
        text = tunnel_log.read_text()
    except Exception:
        continue
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        tunnel_url = m.group()
        break

if not tunnel_url:
    print('ERROR: Tunnel URL not found. Check: !cat /content/cloudflared.log')
else:
    print(f'\n  >>> CLOUDFLARED: {tunnel_url} <<<\n')

print('Starting localtunnel as backup...')
subprocess.run('npm install -g localtunnel', shell=True, capture_output=True)
lt = subprocess.Popen(
    ['lt', '--port', '8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for _ in range(15):
    line = lt.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.loca\.lt', line)
    if m:
        print(f'  >>> LOCALTUNNEL: {m.group()} <<<')
        print(f'\n  (localtunnel will ask for a password — enter this IP:)')
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=5).read().decode().strip()
        print(f'  Password/Endpoint IP: {ip}\n')
        break

print('Try either URL above. ComfyUI is running in background.')

[1/3] ComfyUI started (PID 47474), waiting for port 8188...
[2/3] ComfyUI is ready on port 8188!
[3/3] Cloudflared started, waiting for tunnel URL...

  >>> CLOUDFLARED: https://though-laughing-apr-ending.trycloudflare.com <<<

Starting localtunnel as backup...
  >>> LOCALTUNNEL: https://salty-moles-show.loca.lt <<<

  (localtunnel will ask for a password — enter this IP:)
  Password/Endpoint IP: 34.186.25.219

Try either URL above. ComfyUI is running in background.


## 6. Run the 4-variant pipeline

In [24]:
PRODUCT_FILENAME = 'perfume.png'
BRAND            = 'NOIR'
TAGLINE          = 'Timeless.'
PRODUCT_NAME     = 'perfume bottle'

!python "$PROJECT/scripts/run_pipeline.py" \
    --product "$PROJECT/assets/$PRODUCT_FILENAME" \
    --brand "$BRAND" \
    --tagline "$TAGLINE" \
    --product-name "$PRODUCT_NAME" \
    --server http://127.0.0.1:8188

[1/3] Uploading product image -> http://127.0.0.1:8188
[2/3] Running 4 variants (base seed=2020788512)
      - variant 1/4: A_luxury (Luxury)
      ! variant A_luxury produced no images
      - variant 2/4: B_minimal (Minimal)
      ! variant B_minimal produced no images
      - variant 3/4: C_urban (Urban)
      ! variant C_urban produced no images
      - variant 4/4: D_nostalgic (Nostalgic)
      ! variant D_nostalgic produced no images
[3/3] Skipping text overlay.


## 7. Preview the 4 creatives

In [ ]:
from IPython.display import Image, display
for p in sorted((PROJECT / 'outputs').glob('*.png')):
    if p.name.endswith('_raw.png'):
        continue
    print(p.name)
    display(Image(str(p), width=512))